In [1]:
# Clone the repository
!git clone https://github.com/BardiaAkbari/SymbolicMathematics-LLM

# Move into the specific subfolder containing the script
%cd SymbolicMathematics-LLM/ttrl_lane_emden

Cloning into 'SymbolicMathematics-LLM'...
remote: Enumerating objects: 221, done.
remote: Counting objects: 100% (221/221), done.
remote: Compressing objects: 100% (191/191), done.
remote: Total 221 (delta 72), reused 146 (delta 28), pack-reused 0 (from 0)
Receiving objects: 100% (221/221), 2.51 MiB | 5.31 MiB/s, done.
Resolving deltas: 100% (72/72), done.
/content/SymbolicMathematics-LLM/ttrl_lane_emden


In [2]:
# Check if a requirements file exists and install from it
!if [ -f requirements.txt ]; then pip install -r requirements.txt; fi
!if [ -f ../requirements.txt ]; then pip install -r ../requirements.txt; fi

# Install common libraries needed for Symbolic Mathematics and Transformers/RL
!pip install -q transformers torch datasets accelerate trl sympy scipy matplotlib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 948.2/948.2 kB 11.7 MB/s eta 0:00:00


In [3]:
!wget -q -O ode2.pth https://dl.fbaipublicfiles.com/SymbolicMathematics/models/ode2.pth

!ls -lh ode2.pth

-rw-r--r-- 1 root root 1.8G Mar 14  2020 ode2.pth


In [4]:
!ls

bench_one_step_n5.py   ivp_n5.py		 run_search_baseline.py
bench_reward_batch.py  ode2.pth			 run_ttrl_n2.py
core.py		       run_coverage_sweep.py	 run_ttrl_n5.py
download_ode2.py       run_elite_probe.py	 run_ttrl.py
__init__.py	       run_frozen_n5.py		 smoke_test_n2_ivp.py
inspect_checkpoint.py  run_n2_coverage.py	 smoke_test_n5.py
ivp_n2.py	       run_n2_coverage_sweep.py  smoke_test_verifier.py


In [7]:
!python run_ttrl.py \
  --checkpoint ode2.pth \
  --problem lane_emden \
  --n 0 \
  --lane-form cleared \
  --steps 10 \
  --rollouts 128 \
  --temperature 1.5 \
  --scope last_layer \
  --lr 3e-6

device=cuda
problem=lane_emden_n0_cleared
equation=x*Derivative(f(x), (x, 2)) + x + 2*Derivative(f(x), x)
input_prefix=add x add mul INT+ 2 Y' mul x Y''
expected_reference=a0 + a1/x - x**2/6
checkpoint_arch emb=1024 enc=6 dec=6 heads=8
trainable_parameters=16,889,947 scope=last_layer

[baseline greedy]
reward=26.000 rank=2 residual_zero=True
expr=a8 + a9/x - x**2/6

[baseline fresh eval: 64 samples]
success=59/64 rate=92.188% best_reward=26.000 time(gen=0.55s verify=3.19s)
<lambdifygenerated-67>:2: RuntimeWarning: invalid value encountered in arccosh
  return 2*(-a9 - arccosh(cos((1/4)*log(log(3)))) + (log(2*exp(a8 + x*(a9 + arccosh(cos((1/4)*log(log(3))))))) + exp(5))/x)/x
<lambdifygenerated-83>:2: RuntimeWarning: invalid value encountered in sqrt
  return -(x*(1/4 - a9)/(a9*x - 1/4*x)**2 + (a9*x - 1/4*x)**(-1.0))*sin(x/(a9*x - 1/4*x))/sqrt(cos(x/(a9*x - 1/4*x))) + (1/3)/x - 1/3*(a8 + x)/x**2
<lambdifygenerated-100>:2: RuntimeWarning: invalid value encountered in log
  return x*(-2*a9

In [8]:
!python run_ttrl.py \
  --checkpoint ode2.pth \
  --problem lane_emden \
  --n 1 \
  --lane-form cleared \
  --steps 10 \
  --rollouts 128 \
  --temperature 1.5 \
  --scope last_layer \
  --lr 3e-6

device=cuda
problem=lane_emden_n1_cleared
equation=x*f(x) + x*Derivative(f(x), (x, 2)) + 2*Derivative(f(x), x)
input_prefix=add mul INT+ 2 Y' add mul x Y'' mul x Y
expected_reference=(a0*sin(x) + a1*cos(x))/x
checkpoint_arch emb=1024 enc=6 dec=6 heads=8
trainable_parameters=16,889,947 scope=last_layer

[baseline greedy]
reward=26.000 rank=2 residual_zero=True
expr=cos(a8 + x)/(a9*x + 2*x)

[baseline fresh eval: 64 samples]
success=61/64 rate=95.312% best_reward=26.000 time(gen=0.50s verify=20.63s)
<lambdifygenerated-13>:2: RuntimeWarning: invalid value encountered in sqrt
  return a9*(-((cos(x) + (1 - log(x)**2/(log(x)**2 - 1))*(tan(log(x)/sqrt(1 - log(x)**2))**2 + 1)/(x*sqrt(1 - log(x)**2)))*sin(a8 + sin(x) + tan(log(x)/sqrt(1 - log(x)**2))) - 1)**2*cos(x + cos(a8 + sin(x) + tan(log(x)/sqrt(1 - log(x)**2)))) + ((cos(x) + (1 - log(x)**2/(log(x)**2 - 1))*(tan(log(x)/sqrt(1 - log(x)**2))**2 + 1)/(x*sqrt(1 - log(x)**2)))**2*cos(a8 + sin(x) + tan(log(x)/sqrt(1 - log(x)**2))) - (sin(x) + 2*